# Customer Churn Analysis - Model Training and Evaluation

This notebook trains multiple machine learning models, performs hyperparameter tuning, and evaluates model performance.

## Contents
1. Data Loading and Preparation
2. Baseline Models Training
3. Advanced Models (XGBoost, LightGBM)
4. Hyperparameter Tuning
5. Model Comparison
6. SHAP Explainability
7. Business Impact Analysis
8. Final Model Selection

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.inspection import permutation_importance

from src.config import PIPELINE_PATH, METRICS_PATH, RANDOM_STATE, TEST_SIZE
from src.data_preprocessing import load_raw_features
from src.model_training import train_and_evaluate_pipeline
from src.visualization import (
    plot_confusion_matrix, plot_roc_curve,
    plot_feature_importance, plot_model_comparison
)
from src.utils import explain_model_shap, print_model_summary, format_business_metrics

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Preparation

In [ ]:
# Load raw features; preprocessing happens inside each model's pipeline
print("Loading data...")
X, y = load_raw_features()

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nClass distribution:")
print(y.value_counts(normalize=True))

In [ ]:
# Split data (identical to the split inside train_and_evaluate_pipeline)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nChurn rate in training: {y_train.mean()*100:.2f}%")
print(f"Churn rate in test: {y_test.mean()*100:.2f}%")

## 2. Model Training and Selection

Every model is wrapped in the same preprocessing pipeline and ranked by PR-AUC from 5-fold cross-validation on the training set, so the test set plays no part in choosing the model. The winner's probabilities are then calibrated and its decision threshold is tuned on out-of-fold training predictions. The test set is used once, for the final evaluation.

Running this cell also saves the model, `models/metrics.json` (read by the dashboard) and `reports/model_report.txt`. Pass `tune_hyperparameters=True` to include tuned XGBoost and LightGBM as candidates.

In [ ]:
# Train all models, select, calibrate and save
trainer, results_df = train_and_evaluate_pipeline(X, y, tune_hyperparameters=False)

print_model_summary(results_df)

In [ ]:
# Display results table
results_df.round(4)

## 3. Model Performance Visualization

In [ ]:
# Compare models by different metrics
metrics_to_plot = ['cv_pr_auc', 'precision', 'recall', 'f1_score']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    results_sorted = results_df.sort_values(metric, ascending=False)
    ax = axes[idx]

    bars = ax.barh(results_sorted['model'], results_sorted[metric], color='steelblue')

    # Color the best model
    bars[0].set_color('coral')

    ax.set_xlabel(metric.replace('_', ' ').title(), fontsize=12)
    ax.set_title(f'Model Comparison - {metric.replace("_", " ").title()}',
                 fontsize=14, fontweight='bold')
    ax.invert_yaxis()

    # Add value labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2,
                f'{width:.3f}', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ROC-AUC comparison
if 'roc_auc' in results_df.columns:
    plot_model_comparison(results_df, metric='roc_auc', save=True)

## 4. Best Model Evaluation

In [ ]:
# Final model: the selected model with calibrated probabilities
best_model_name = trainer.best_model_name
best_model = trainer.final_model
threshold = trainer.threshold

print(f"Best Model: {best_model_name} (calibrated)")
print(f"Decision threshold: {threshold:.3f}")

In [ ]:
# Predictions at the tuned threshold
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= threshold).astype(int)

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion Matrix
plot_confusion_matrix(
    y_test, y_pred,
    labels=['No Churn', 'Churn'],
    title=f'Confusion Matrix - {best_model_name}',
    save=True,
    save_name='confusion_matrix_best_model.png'
)

In [ ]:
# ROC Curve
plot_roc_curve(
    y_test, y_pred_proba,
    model_name=best_model_name,
    save=True,
    save_name='roc_curve_best_model.png'
)

## 5. Feature Importance Analysis

In [ ]:
# Permutation importance on the raw input columns: how much test-set PR-AUC drops
# when each column is shuffled. Works for any model, including the calibrated one.
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring='average_precision', n_repeats=5, random_state=RANDOM_STATE
)
feature_names = X_test.columns.tolist()
importances = perm.importances_mean

plot_feature_importance(
    feature_names, importances,
    top_n=20,
    title=f'Permutation Importance - {best_model_name}',
    save=True,
    save_name='feature_importance_best_model.png'
)

# Display top features
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feat_imp_df.head(10).to_string(index=False))

## 6. SHAP Explainability Analysis

In [ ]:
# SHAP Analysis
# SHAP explains the selected model before calibration. Isotonic calibration only
# rescales its scores monotonically, so the feature effects carry over.
print("Generating SHAP explanations...")
print("This may take a few minutes...\n")

selected_pipeline = trainer.best_model
preprocess = selected_pipeline.named_steps['preprocess']
classifier = selected_pipeline.named_steps['classifier']

X_train_transformed = preprocess.transform(X_train)
X_test_transformed = preprocess.transform(X_test)
shap_feature_names = preprocess.named_steps['columns'].get_feature_names_out().tolist()

# Summary plot
explainer = explain_model_shap(
    classifier,
    X_train_transformed[:100],  # Use subset for faster computation
    X_test_transformed[:100],
    shap_feature_names,
    plot_type='summary',
    max_display=20,
    save=True
)

In [ ]:
# SHAP bar plot
explainer = explain_model_shap(
    classifier,
    X_train_transformed[:100],
    X_test_transformed[:100],
    shap_feature_names,
    plot_type='bar',
    max_display=20,
    save=True
)

## 7. Hyperparameter Tuning (Optional)

Tuning runs inside `train_and_evaluate_pipeline`. Re-run section 2 with `tune_hyperparameters=True` to add tuned XGBoost and LightGBM as candidates; a tuned model replaces the selected one only if its cross-validated PR-AUC is higher.

## 8. Business Impact Analysis

In [ ]:
# Calculate business impact (assumptions are set in src/config.py)
business_metrics = trainer.calculate_business_impact(y_test, y_pred)

print(format_business_metrics(business_metrics))

In [ ]:
# Visualize business impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Outcome for actual churners
outcomes = [
    business_metrics['customers_saved'],
    business_metrics['true_positives'] - business_metrics['customers_saved'],
    business_metrics['false_negatives']
]
labels = ['Saved by Campaign', 'Contacted but Left', 'Missed by Model']
colors = ['#38ada9', '#ffb38a', '#ff6b6b']

axes[0].bar(labels, outcomes, color=colors)
axes[0].set_title('Outcome for Actual Churners', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Customers', fontsize=12)
for i, v in enumerate(outcomes):
    axes[0].text(i, v, f'{v:.0f}', ha='center', va='bottom', fontsize=10)

# Net benefit vs. no campaign
categories = ['Retention Cost', 'Value Retained', 'Net Benefit']
values = [
    -business_metrics['cost_of_retention'],
    business_metrics['value_retained'],
    business_metrics['net_benefit']
]
colors_benefit = ['#ff6b6b', '#95e1d3', '#38ada9']

bars = axes[1].bar(categories, values, color=colors_benefit)
axes[1].set_title('Financial Summary vs. No Campaign', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Amount ($)', fontsize=12)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

for i, (bar, v) in enumerate(zip(bars, values)):
    axes[1].text(bar.get_x() + bar.get_width()/2, v,
                f'${v:,.0f}', ha='center',
                va='bottom' if v > 0 else 'top', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Saved Outputs

In [ ]:
# Section 2 already saved the model, the dashboard metrics and the text report
print(f"Model (preprocessing + calibrated {best_model_name}): {PIPELINE_PATH}")
print(f"Dashboard metrics: {METRICS_PATH}")
print("Text report: reports/model_report.txt")

## 10. Model Deployment Recommendations

In [ ]:
# Churn rates behind the insights, computed from the raw data
high_cs_calls = X['Customer service calls'] >= 4
intl_plan = X['International plan'] == 'Yes'

print("\n" + "="*80)
print("MODEL DEPLOYMENT RECOMMENDATIONS")
print("="*80)

print(f"\n1. BEST MODEL: {best_model_name} (calibrated, threshold {threshold:.3f})")
print(f"   - CV PR-AUC (selection metric): {results_df.iloc[0]['cv_pr_auc']:.4f}")
print(f"   - Test precision: {precision_score(y_test, y_pred):.4f}")
print(f"   - Test recall: {recall_score(y_test, y_pred):.4f}")

print("\n2. KEY INSIGHTS:")
print(f"   - Most important inputs: {', '.join(feat_imp_df['feature'].head(3))}")
print(f"   - Customers with 4+ customer service calls churn at {y[high_cs_calls].mean():.0%}, "
      f"versus {y[~high_cs_calls].mean():.0%} for everyone else")
print(f"   - International plan customers churn at {y[intl_plan].mean():.0%}, "
      f"versus {y[~intl_plan].mean():.0%} without one")
print(f"   - Account length has almost no relationship with churn "
      f"(correlation {X['Account length'].corr(y):.2f})")

print("\n3. BUSINESS RECOMMENDATIONS:")
print("   - Reach out after a customer's 3rd service call, before frustration turns into churn")
print("   - Review international plan pricing and call quality, since those customers churn far more often")
print("   - Watch heavy daytime users, whose high total charges make them more likely to leave")

print("\n4. DEPLOYMENT STRATEGY:")
print("   - Deploy model via Streamlit dashboard")
print("   - Set up automated weekly predictions")
print("   - Monitor model performance monthly")
print("   - Retrain model quarterly with new data")
print("   - A/B test retention strategies")

print("\n" + "="*80)
print("MODELING COMPLETE! Ready for dashboard deployment.")
print("="*80)